# Workload to inference decision

This notebook turns explicit buyer assumptions into raw request evidence, capacity and cost estimates, and a verified decision report. The benchmark is labeled **simulated**: it validates the pipeline and does not claim model performance.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from reka_inference_lab.artifacts import verify_bundle
from reka_inference_lab.benchmark import simulate_benchmark
from reka_inference_lab.capacity import build_capacity_plan
from reka_inference_lab.models import Pricing, PromptCase, Workload, read_json
from reka_inference_lab.report import write_report_bundle

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
prompts_data = read_json(root / "examples/prompts.json")
prompts = [PromptCase.from_mapping(row) for row in prompts_data["prompts"]]
workload = Workload.from_mapping(read_json(root / "examples/workload.json"))
pricing = Pricing.from_mapping(read_json(root / "examples/reka_public_pricing_2026-08-04.json"))
len(prompts), workload.name, pricing.checked_at

## Generate raw evidence

The simulator exposes its assumptions and returns the same raw-record schema used by observed endpoint and local runs.

In [ ]:
benchmark = simulate_benchmark(
    prompts, request_count=24, concurrency=4, model="reka-edge-evaluation-shape"
)
assert benchmark["evidence_type"] == "simulated"
summary = benchmark["summary"]
{
    "success_rate": summary["success_rate"],
    "p95_e2e_ms": round(summary["e2e_ms"]["p95"], 1),
    "output_tokens_per_second": round(summary["output_tokens_per_second"], 1),
}

## Translate evidence into a bounded decision

A capacity unit means one endpoint/configuration matching the benchmark. It is not assumed to equal one GPU.

In [ ]:
plan = build_capacity_plan(benchmark, workload, pricing)
with TemporaryDirectory() as temporary:
    paths = write_report_bundle(temporary, benchmark, plan)
    verified = verify_bundle(temporary)
    assert verified["ok"]
    report_preview = Path(paths["markdown_report"]).read_text()[:500]
verified, report_preview

## Run the repository tests

The notebook finishes by invoking the same standard-library test suite used by the release gate.

In [ ]:
import subprocess
import sys

completed = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", str(root / "tests"), "-v"],
    cwd=root,
    check=True,
    capture_output=True,
    text=True,
)
assert "OK" in completed.stderr
print("Repository tests passed from the executable notebook.")